In [ ]:
from datetime import datetime, timezone
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"

from aitana import whakaari

from whakaaribn import (pre_eruption_window,
                        get_color)


In [ ]:
try:
    data_file = snakemake.input[0]
    pio.get_chrome()
except NameError:
    data_file = '../data/whakaari_data_with_groups.csv'

In [ ]:
fig = make_subplots(rows=11, cols=1, shared_xaxes=True, vertical_spacing=0.02,
                    specs=[[{}], 
                           [{"rowspan": 2, "secondary_y": True}],
                           [{}], 
                           [{"rowspan": 2, "secondary_y": True}],
                           [{}],
                           [{"rowspan": 2, "secondary_y": True}], 
                           [{}], 
                           [{"rowspan": 2, "secondary_y": True}],
                           [{}],
                           [{"rowspan": 2, "secondary_y": True}],
                           [{}]],
                    start_cell="bottom-left")
plot_dict = {'RSAM': dict(name='RSAM [nm/s]', mode='lines'),
             'Eqr': dict(name='Eq. rate [1/day]', mode='lines'),
             'CO2': dict(name=u'CO\u2082 [t/day]', mode='markers'),
             'SO2': dict(name=u'SO\u2082 [t/day]', mode='markers'),
             'H2S': dict(name=u'H\u2082S [t/day]', mode='markers')}


data = pd.read_csv(data_file, parse_dates=True, index_col=0)
data['eruptions'] = pre_eruption_window(data['eruptions'], 30)
for i, g in enumerate(data.groupby('group')):
    fig.add_trace(go.Scatter(x=g[1].index, y=g[1]['eruptions'], mode='lines', line_color=get_color(i+2),
                             showlegend=False, legendgroup='group1', legendgrouptitle_text='Data splits',
                             name='Group %s' % g[0]), row=1, col=1)
seismic_end_date = datetime(2022, 8, 4, tzinfo=timezone.utc)
for i, col in enumerate(plot_dict.keys()):
    showlegend = True
    if i > 0:
        showlegend = False
    _x = data.index
    _y = data[col].ffill()
    _y_raw = data[col]
    if col in ['RSAM', 'Eqr']:
        _x = _x[_x <= seismic_end_date]
        _y = _y.iloc[:len(_x)]
        _y_raw = data[col].iloc[:len(_x)]
    fig.add_trace(go.Scatter(x=_x, y=_y, mode='lines', line_color=get_color(1),
                             showlegend=showlegend, name='Imputed data', legendgroup='group',
                             legendgrouptitle_text="Input data"), row=i*2+2, col=1, secondary_y=False)
    color = get_color(0)
    if plot_dict[col]['mode'] == 'markers':
        color = get_color(0, alpha=0.7)
    fig.add_trace(go.Scatter(x=_x, y=_y_raw, mode=plot_dict[col]['mode'],
                             showlegend=showlegend, name='Raw data', legendgroup='group',
                             legendgrouptitle_text="Input data", line_color=color), row=i*2+2, col=1,
                    secondary_y=False)
    
    fig.update_yaxes(title=plot_dict[col]['name'], row=i*2+2, col=1, secondary_y=False)
if True:
    eruptions = whakaari.eruptions(end_date=data.index[-1]) 
    dfe_ = eruptions.loc["2004-01-01":]
    showlegend = True 
    for row in [2, 4, 6, 8, 10]:
        for i in range(len(dfe_.index)):
            fig.add_trace(go.Scatter(x=[dfe_.index[i], dfe_.index[i]], y=[0., 1.], mode='lines',
                                     line_width=.8, line_color='black', name='Observed eruption', 
                                     showlegend=showlegend), secondary_y=True, row=row, col=1)
            showlegend = False

fig.update_layout(height=1000, width=1200)
fig.update_layout(legend=dict(y=1.15, orientation='h'))
fig.update_yaxes(type='log', nticks=3, secondary_y=False)
fig.update_yaxes(type='linear', row=1, col=1)
fig.update_yaxes(dtick=1, row=1, col=1)
x_annot= "2009-01-01"
y_annot = 0.9
fig.add_annotation(text="<b>(F)</b>", xref="x", yref="y domain", x=x_annot, y=y_annot, showarrow=False)
fig.add_annotation(text="<b>(E)</b>", xref="x", yref="y2 domain", x=x_annot, y=y_annot, showarrow=False)
fig.add_annotation(text="<b>(D)</b>", xref="x", yref="y5 domain", x=x_annot, y=y_annot, showarrow=False)
fig.add_annotation(text="<b>(C)</b>", xref="x", yref="y8 domain", x=x_annot, y=y_annot, showarrow=False)
fig.add_annotation(text="<b>(B)</b>", xref="x", yref="y11 domain", x=x_annot, y=y_annot, showarrow=False)
fig.add_annotation(text="<b>(A)</b>", xref="x", yref="y14 domain", x=x_annot, y=y_annot, showarrow=False)
fig.add_annotation(text="<b>Pre-eruption window</b>", xref="x", yref="y domain", x="2010-09-01", y=y_annot, showarrow=False)
fig.add_annotation(text="<b>Data groups:</b>", xref="x", yref="y domain", x="2010-01-01", y=.2, showarrow=False)
fig.add_annotation(text="<b>Group A</b>", xref="x", yref="y domain", x="2011-10-01", y=.2, showarrow=False)
fig.add_annotation(text="<b>Group B</b>", xref="x", yref="y domain", x="2013-03-01", y=.2, showarrow=False)
fig.add_annotation(text="<b>Group C</b>", xref="x", yref="y domain", x="2015-06-01", y=.2, showarrow=False)
fig.add_annotation(text="<b>Group D</b>", xref="x", yref="y domain", x="2018-03-01", y=.2, showarrow=False)
fig.add_annotation(text="<b>Group E</b>", xref="x", yref="y domain", x="2022-02-01", y=.2, showarrow=False)
fig.update_yaxes(showticklabels=False, showgrid=False, secondary_y=True)
try:
    fig.write_image(snakemake.output[0], height=1000, width=1200, scale=3)
except NameError:
    pass
fig